# ISIC 2018 — Mole Network Structure Analysis: EDA
> **Belle AI Technical Test — Part 1: Data Exploration & Analysis**

This notebook covers:
1. Dataset paths & helper functions
2. Dataset overview (counts, splits)
3. Image statistics (resolution, color)
4. Annotation analysis (5 dermoscopic structures)
5. Class imbalance & co-occurrence
6. Annotation quality assessment
7. Key findings & implications for model design

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Libraries loaded ✓')

## 1. Dataset Paths & Helper Functions

In [ ]:
# ─── Raw paths — giữ nguyên cấu trúc gốc của ISIC ────────────────────────────
# Sửa tên folder cho khớp với tên dataset bạn upload lên Kaggle
BASE = Path('/kaggle/input')

TRAIN_IMG_DIR  = BASE / 'ISIC2018_Task1-2_Training_Input'
VAL_IMG_DIR    = BASE / 'ISIC2018_Task1-2_Validation_Input'
TRAIN_MASK_DIR = BASE / 'ISIC2018_Task2_Training_GroundTruth_v3'
VAL_MASK_DIR   = BASE / 'ISIC2018_Task2_Validation_GroundTruth'

ATTRIBUTES = [
    'pigment_network',
    'negative_network',
    'streaks',
    'milia_like_cyst',
    'globules'
]

ATTR_COLORS = {
    'pigment_network' : '#E74C3C',
    'negative_network': '#3498DB',
    'streaks'         : '#2ECC71',
    'milia_like_cyst' : '#F39C12',
    'globules'        : '#9B59B6'
}

# ─── Helper functions — dùng xuyên suốt toàn bộ project ──────────────────────

def get_image_ids(split: str = 'train') -> list:
    """Trả về list tất cả image IDs của split."""
    d = TRAIN_IMG_DIR if split == 'train' else VAL_IMG_DIR
    return sorted([p.stem for p in d.glob('*.jpg')])

def get_image_path(image_id: str, split: str = 'train') -> Path:
    """Trả về Path ảnh gốc."""
    d = TRAIN_IMG_DIR if split == 'train' else VAL_IMG_DIR
    return d / f'{image_id}.jpg'

def get_mask_path(image_id: str, attribute: str, split: str = 'train') -> Path:
    """Trả về Path mask của 1 attribute."""
    d = TRAIN_MASK_DIR if split == 'train' else VAL_MASK_DIR
    return d / f'{image_id}_attribute_{attribute}.png'

def load_image(image_id: str, split: str = 'train') -> np.ndarray:
    """Load ảnh gốc → numpy array RGB."""
    return np.array(Image.open(get_image_path(image_id, split)).convert('RGB'))

def load_mask(image_id: str, attribute: str, split: str = 'train') -> np.ndarray:
    """Load binary mask (0/1) của 1 attribute. Trả về None nếu không tồn tại."""
    p = get_mask_path(image_id, attribute, split)
    if not p.exists():
        return None
    return (np.array(Image.open(p).convert('L')) > 127).astype(np.uint8)

def load_all_masks(image_id: str, split: str = 'train') -> dict:
    """Load cả 5 masks → dict {attribute: np.ndarray}."""
    return {attr: load_mask(image_id, attr, split) for attr in ATTRIBUTES}


# ─── Sanity check ─────────────────────────────────────────────────────────────
print('Sanity check:')
for split in ['train', 'val']:
    ids = get_image_ids(split)
    print(f'\n[{split}] {len(ids)} images')
    sample_id = ids[0]
    for attr in ATTRIBUTES:
        p = get_mask_path(sample_id, attr, split)
        status = '✓' if p.exists() else '✗ MISSING'
        print(f'  {status}  masks/{split}/{p.name}')

## 2. Dataset Overview

In [ ]:
train_ids = get_image_ids('train')
val_ids   = get_image_ids('val')

print('=== Dataset Overview ===')
print(f'Training   images : {len(train_ids)}')
print(f'           masks  : {len(train_ids)} × 5 attributes = {len(train_ids)*5} files')
print(f'Validation images : {len(val_ids)}')
print(f'           masks  : {len(val_ids)} × 5 attributes = {len(val_ids)*5} files')
print(f'Total images      : {len(train_ids) + len(val_ids)}')
print()
print('Dermoscopic structures (Task 2):')
for i, attr in enumerate(ATTRIBUTES, 1):
    print(f'  {i}. {attr}')

## 3. Image Statistics

In [ ]:
np.random.seed(42)
sample_ids = np.random.choice(train_ids, size=min(300, len(train_ids)), replace=False)

widths, heights = [], []
for sid in tqdm(sample_ids, desc='Reading image sizes'):
    w, h = Image.open(get_image_path(sid)).size
    widths.append(w)
    heights.append(h)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths,  bins=30, color='#3498DB', edgecolor='white')
axes[0].set_title('Width Distribution')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=30, color='#2ECC71', edgecolor='white')
axes[1].set_title('Height Distribution')
axes[1].set_xlabel('Height (px)')

axes[2].scatter(widths, heights, alpha=0.3, s=10, color='#9B59B6')
axes[2].set_title('Width vs Height')
axes[2].set_xlabel('Width (px)')
axes[2].set_ylabel('Height (px)')

plt.suptitle('Image Resolution Analysis (n=300 sample)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

## 4. Annotation Analysis — 5 Dermoscopic Structures

In [ ]:
records = []
for sid in tqdm(train_ids, desc='Analyzing masks'):
    row = {'image_id': sid}
    for attr in ATTRIBUTES:
        mask = load_mask(sid, attr, 'train')
        if mask is not None:
            row[f'{attr}_present']  = int(mask.sum() > 0)
            row[f'{attr}_coverage'] = mask.sum() / mask.size
        else:
            row[f'{attr}_present']  = 0
            row[f'{attr}_coverage'] = 0.0
    records.append(row)

df = pd.DataFrame(records)
df['num_structures'] = df[[f'{a}_present' for a in ATTRIBUTES]].sum(axis=1)
print(f'DataFrame: {df.shape[0]} images × {df.shape[1]} columns')
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Prevalence
pcts = [df[f'{a}_present'].mean()*100 for a in ATTRIBUTES]
bars = axes[0].bar(ATTRIBUTES, pcts,
                   color=[ATTR_COLORS[a] for a in ATTRIBUTES], edgecolor='white')
axes[0].set_title('Structure Prevalence (%)', fontweight='bold')
axes[0].set_ylabel('%')
axes[0].set_xticklabels(ATTRIBUTES, rotation=20, ha='right')
axes[0].axhline(50, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars, pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5, f'{val:.1f}%', ha='center', fontsize=10)

# Structures per image
vc = df['num_structures'].value_counts().sort_index()
axes[1].bar(vc.index, vc.values, color='#34495E', edgecolor='white')
axes[1].set_title('# Structures per Image', fontweight='bold')
axes[1].set_xlabel('Number of Structures Present')
axes[1].set_ylabel('Number of Images')
for x, y in zip(vc.index, vc.values):
    axes[1].text(x, y + 5, str(y), ha='center', fontsize=10)

plt.suptitle('Class Distribution — ISIC 2018 Task 2 (Training)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Prevalence:')
for attr, pct in zip(ATTRIBUTES, pcts):
    tag = 'RARE' if pct < 20 else ('COMMON' if pct > 60 else 'MODERATE')
    print(f'  {attr:20s}: {pct:5.1f}%  [{tag}]')

In [ ]:
# Coverage distribution khi structure có mặt
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, attr in enumerate(ATTRIBUTES):
    cov = df[df[f'{attr}_present'] == 1][f'{attr}_coverage'] * 100
    axes[i].hist(cov, bins=30, color=ATTR_COLORS[attr], edgecolor='white')
    axes[i].set_title(attr.replace('_', '\n'), fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Coverage (%)')
    axes[i].axvline(cov.mean(), color='black', linestyle='--', lw=1.5,
                    label=f'μ={cov.mean():.1f}%')
    axes[i].legend(fontsize=8)
    if i == 0:
        axes[i].set_ylabel('Count')

plt.suptitle('Mask Coverage (% image area) when Structure is Present',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Co-occurrence Analysis

In [ ]:
cooccur = pd.DataFrame(index=ATTRIBUTES, columns=ATTRIBUTES, dtype=float)
for a in ATTRIBUTES:
    for b in ATTRIBUTES:
        both   = ((df[f'{a}_present']==1) & (df[f'{b}_present']==1)).sum()
        either = ((df[f'{a}_present']==1) | (df[f'{b}_present']==1)).sum()
        cooccur.loc[a, b] = both / either if either > 0 else 0

short = ['pigment\nnet', 'negative\nnet', 'streaks', 'milia\ncyst', 'globules']
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cooccur.astype(float), annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=short, yticklabels=short, ax=ax, linewidths=0.5, vmin=0, vmax=1)
ax.set_title('Structure Co-occurrence (Jaccard Similarity)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Sample Visualization

In [ ]:
def visualize_sample(image_id: str, split: str = 'train'):
    """Hiển thị ảnh gốc + overlay 5 masks."""
    img = load_image(image_id, split)
    fig, axes = plt.subplots(1, 6, figsize=(20, 3))

    axes[0].imshow(img)
    axes[0].set_title('Original', fontweight='bold')
    axes[0].axis('off')

    for j, attr in enumerate(ATTRIBUTES):
        mask = load_mask(image_id, attr, split)
        axes[j+1].imshow(img)
        if mask is not None and mask.sum() > 0:
            overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
            hex_c = ATTR_COLORS[attr].lstrip('#')
            rgb = [int(hex_c[k:k+2], 16)/255 for k in (0, 2, 4)]
            overlay[mask == 1] = rgb + [0.5]
            axes[j+1].imshow(overlay)
            cov = mask.sum() / mask.size * 100
            axes[j+1].set_title(f'{attr.replace("_", chr(10))}\n{cov:.2f}%', fontsize=8)
        else:
            axes[j+1].set_title(f'{attr.replace("_", chr(10))}\n(absent)',
                                 fontsize=8, color='gray')
        axes[j+1].axis('off')

    plt.suptitle(f'ID: {image_id}', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

np.random.seed(7)
for sid in np.random.choice(train_ids, size=4, replace=False):
    visualize_sample(sid)

## 7. Annotation Quality Assessment

In [ ]:
print(f'=== Annotation Quality Flags (Training Set) ===')
print(f'{"Attribute":20s} | {"Present":>8} | {"Absent":>8} | {"Tiny (<0.1%)":>13} | {"Huge (>80%)":>11}')
print('-' * 70)

for attr in ATTRIBUTES:
    present_df = df[df[f'{attr}_present'] == 1]
    absent_n   = len(df) - len(present_df)
    cov = present_df[f'{attr}_coverage']
    tiny = (cov < 0.001).sum()
    huge = (cov > 0.80).sum()
    print(f'{attr:20s} | {len(present_df):>8} | {absent_n:>8} | {tiny:>13} | {huge:>11}')

print()
print('Notes:')
print('  Tiny masks (<0.1%) → potential annotation noise or borderline cases')
print('  Huge masks (>80%)  → possible over-annotation')

## 8. Summary Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pcts     = [df[f'{a}_present'].mean()*100 for a in ATTRIBUTES]
avg_covs = [df[df[f'{a}_present']==1][f'{a}_coverage'].mean()*100 for a in ATTRIBUTES]
ratios   = [df[f'{a}_present'].mean() / (1-df[f'{a}_present'].mean()+1e-6) for a in ATTRIBUTES]
colors   = [ATTR_COLORS[a] for a in ATTRIBUTES]

for ax, vals, title, xlabel, vline in [
    (axes[0], pcts,     'Prevalence (%)',               '%',               50),
    (axes[1], avg_covs, 'Avg Mask Coverage\n(when present)', '% of image', None),
    (axes[2], ratios,   'Imbalance Ratio\n(pos/neg)',   'ratio',           1.0),
]:
    ax.barh(ATTRIBUTES, vals, color=colors, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(xlabel)
    if vline:
        ax.axvline(vline, color='gray', linestyle='--', alpha=0.5)
    for i, v in enumerate(vals):
        ax.text(v + max(vals)*0.01, i, f'{v:.1f}', va='center', fontsize=9)

plt.suptitle('ISIC 2018 Task 2 — EDA Summary Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved eda_summary.png ✓')

## 9. Summary Table & Key Findings

In [ ]:
rows = []
for attr in ATTRIBUTES:
    present_df = df[df[f'{attr}_present'] == 1]
    cov = present_df[f'{attr}_coverage'] * 100
    rows.append({
        'Structure'       : attr,
        'Prevalence (%)'  : f"{df[f'{attr}_present'].mean()*100:.1f}",
        'N present'       : len(present_df),
        'N absent'        : len(df) - len(present_df),
        'Avg Cov (%)'     : f"{cov.mean():.2f}",
        'Median Cov (%)'  : f"{cov.median():.2f}",
        'Std Cov (%)'     : f"{cov.std():.2f}",
    })

print(pd.DataFrame(rows).to_string(index=False))

## 10. Key Findings & Implications for Model Design

### Findings

**1. Severe class imbalance**  
`pigment_network` is the most common (~50–60%). `streaks`, `milia_like_cyst`, `negative_network` are significantly rarer (<20%). A naive model will ignore minority classes.

**2. Highly variable mask coverage**  
`pigment_network` can cover >30% of image area. `streaks` are thin and elongated, typically <5%. This means different structures require different receptive field sizes.

**3. Annotation noise**  
Masks flagged as 'present' but with <0.1% coverage are likely annotation errors or ambiguous borderline cases. Some visually present structures may also be left unannotated (annotator fatigue).

**4. Co-occurrence patterns**  
`pigment_network` and `negative_network` tend to co-occur — they are structurally related. A shared encoder can exploit this correlation.

**5. Many zero-structure images**  
A notable portion of images have 0 annotated structures, making them pure background samples.

### Implications for Model Design

| Issue | Mitigation |
|---|---|
| Class imbalance | Weighted BCE / Focal Loss per attribute |
| Thin structures (streaks) | Dice Loss + boundary-aware loss |
| Annotation noise | Label smoothing, ignore very sparse masks |
| Variable scale | Multi-scale decoder (FPN / UNet++) |
| Many 0-structure images | Hard negative mining or balanced sampling |
| Co-occurrence | Multi-task segmentation with shared encoder |